# Data Mining Assignment: Spark Data Lake Preprocessing
**Dataset:** `sales.csv` &nbsp;|&nbsp; **Environment:** Google Colab + PySpark

Workflow: **Bronze** (raw, untouched) → inspect with Spark → basic preprocessing → **Silver** (cleaned output).
No Gold layer or detailed analysis is performed. All discrepancy counts below are computed with Spark, nothing is hard-coded.

**How to run:** open in Google Colab, choose *Runtime → Run all*. When the upload prompt appears, select `sales.csv`.

## 1. Setup: install PySpark and create the Data Lake folders

In [1]:
!pip install -q pyspark

In [2]:
import os, shutil, hashlib

BASE   = "data_lake"
BRONZE = f"{BASE}/bronze"
SILVER = f"{BASE}/silver"
SILVER_PARQUET = f"{SILVER}/preprocessed_sales"

for d in (BRONZE, SILVER_PARQUET):
    os.makedirs(d, exist_ok=True)

# Get sales.csv into the Colab runtime (skip the upload if it is already here)
if not os.path.exists("sales.csv"):
    try:
        from google.colab import files
        uploaded = files.upload()                    # choose sales.csv
        name = next(iter(uploaded))
        if name != "sales.csv":
            os.rename(name, "sales.csv")
    except ImportError:
        raise FileNotFoundError("Place sales.csv next to this notebook.")

# Byte-for-byte copy into Bronze (raw data must stay exactly as provided)
shutil.copyfile("sales.csv", f"{BRONZE}/sales.csv")

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
print("Bronze identical to original:", md5("sales.csv") == md5(f"{BRONZE}/sales.csv"))

for root, dirs, fs in os.walk(BASE):
    print(root + "/", fs)

Bronze identical to original: True
data_lake/ []
data_lake/bronze/ ['sales.csv']
data_lake/silver/ []
data_lake/silver/preprocessed_sales/ []


## 2. Load and inspect the raw data with Spark
All columns are read as **strings** (`inferSchema` off) on purpose: this keeps malformed numbers and dates visible instead of letting Spark silently turn them into nulls. Typing is done later in the Silver step.

In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("SalesDataLake")
         .master("local[*]")
         .config("spark.sql.ansi.enabled", "false")              # bad casts/dates -> null instead of errors
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .getOrCreate())

bronze = spark.read.option("header", True).option("inferSchema", False).csv(f"{BRONZE}/sales.csv")

print("Schema:")
bronze.printSchema()
print("First rows:")
bronze.show(8, truncate=False)
raw_count = bronze.count()
print("Total records in Bronze:", raw_count)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/13 12:05:46 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/13 12:05:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/13 12:05:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

First rows:


+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree        |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt      |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie       |Clothing |6       |3080      |12              |UPI             |Pune      |Maharashtra   |2025-05-06|Shipped     

Total records in Bronze: 1000


### 2.1 Duplicates

In [4]:
distinct_count = bronze.distinct().count()
print("Exact duplicate rows (extra copies):", raw_count - distinct_count)

print("Rows involved in exact duplication:")
(bronze.groupBy(bronze.columns).count().where("count > 1")
       .orderBy("order_id").show(20, truncate=False))

# order_id repeated but with different content (NOT exact duplicates)
conflicts = (bronze.dropDuplicates().groupBy("order_id").count().where("count > 1"))
print("order_ids that repeat with different row content:", conflicts.count())

Exact duplicate rows (extra copies): 10
Rows involved in exact duplication:


+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100055  |C0270      |Pranav Iyer     |Dry Fruits Pack  |Grocery        |7       |930       |12              |Cash on Delivery|Coimbatore|Tamil Nadu    |2026-08-20|Delivered   |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack  |Grocery        |7      

order_ids that repeat with different row content: 8


### 2.2 Null / blank values

In [5]:
is_blank = lambda c: F.col(c).isNull() | (F.trim(F.col(c)) == "")
bronze.select([F.sum(F.when(is_blank(c), 1).otherwise(0)).alias(c) for c in bronze.columns]).show(vertical=True)

-RECORD 0---------------
 order_id         | 0   
 customer_id      | 0   
 customer_name    | 10  
 product          | 0   
 category         | 0   
 quantity         | 0   
 unit_price       | 0   
 discount_percent | 0   
 payment_method   | 8   
 city             | 8   
 state            | 0   
 order_date       | 0   
 order_status     | 0   



### 2.3 Text-format inconsistencies (extra spaces, mixed capitalization)

In [6]:
text_cols = ["customer_id", "customer_name", "product", "category",
             "payment_method", "city", "state", "order_status"]

# (a) leading/trailing/repeated spaces
spaces = bronze.select([
    F.sum(F.when(F.col(c).isNotNull() & (F.col(c) != F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")), 1).otherwise(0)).alias(c)
    for c in text_cols])
print("Rows with extra whitespace, per column:")
spaces.show(vertical=True)

# (b) same value spelled with different capitalization
print("Capitalization variants:")
for c in text_cols:
    variants = (bronze.where(F.col(c).isNotNull())
                .select(F.trim(F.col(c)).alias("v")).distinct()
                .groupBy(F.lower("v").alias("key"))
                .agg(F.collect_set("v").alias("spellings"))
                .where(F.size("spellings") > 1))
    if variants.count():
        print(f"--- {c}")
        variants.show(truncate=False)

Rows with extra whitespace, per column:


-RECORD 0-------------
 customer_id    | 0   
 customer_name  | 3   
 product        | 0   
 category       | 0   
 payment_method | 0   
 city           | 3   
 state          | 0   
 order_status   | 0   

Capitalization variants:


--- category


+-----------+--------------------------+
|key        |spellings                 |
+-----------+--------------------------+
|beauty     |[beauty, Beauty]          |
|electronics|[ELECTRONICS, Electronics]|
|sports     |[sports, Sports, SPORTS]  |
|clothing   |[CLOTHING, Clothing]      |
+-----------+--------------------------+



--- city


+----------+------------------------+
|key       |spellings               |
+----------+------------------------+
|delhi     |[DELHI, delhi, Delhi]   |
|hyderabad |[HYDERABAD, Hyderabad]  |
|jaipur    |[Jaipur, JAIPUR]        |
|coimbatore|[Coimbatore, coimbatore]|
+----------+------------------------+



### 2.4 Invalid numerical values

In [7]:
num = bronze.select(
    "order_id",
    F.col("quantity").cast("int").alias("q"),
    F.col("unit_price").cast("double").alias("p"),
    F.col("discount_percent").cast("double").alias("d"),
    "quantity", "unit_price", "discount_percent")

num.agg(
    F.sum(F.when(F.col("quantity").isNotNull() & F.col("q").isNull(), 1).otherwise(0)).alias("quantity_non_numeric"),
    F.sum(F.when(F.col("q") <= 0, 1).otherwise(0)).alias("quantity_le_0"),
    F.sum(F.when(F.col("unit_price").isNotNull() & F.col("p").isNull(), 1).otherwise(0)).alias("unit_price_non_numeric"),
    F.sum(F.when(F.col("p") <= 0, 1).otherwise(0)).alias("unit_price_le_0"),
    F.sum(F.when(F.col("discount_percent").isNotNull() & F.col("d").isNull(), 1).otherwise(0)).alias("discount_non_numeric"),
    F.sum(F.when((F.col("d") < 0) | (F.col("d") > 100), 1).otherwise(0)).alias("discount_outside_0_100"),
).show(vertical=True)

print("Examples of invalid numeric values:")
num.where((F.col("q") <= 0) | (F.col("p") <= 0) | (F.col("d") < 0) | (F.col("d") > 100)) \
   .select("order_id", "quantity", "unit_price", "discount_percent").show(10)

-RECORD 0---------------------
 quantity_non_numeric   | 0   
 quantity_le_0          | 12  
 unit_price_non_numeric | 0   
 unit_price_le_0        | 8   
 discount_non_numeric   | 0   
 discount_outside_0_100 | 8   

Examples of invalid numeric values:


+--------+--------+----------+----------------+
|order_id|quantity|unit_price|discount_percent|
+--------+--------+----------+----------------+
|  100034|       7|      1720|              -5|
|  100102|       7|         0|              40|
|  100114|       0|      2920|              15|
|  100140|       4|       790|              -5|
|  100143|       0|      1030|              15|
|  100170|       6|         0|               0|
|  100198|      -2|      2160|              12|
|  100217|       1|       650|              -5|
|  100235|       7|      -500|               0|
|  100284|       2|      -500|              30|
+--------+--------+----------+----------------+
only showing top 10 rows


### 2.5 Inconsistent / invalid dates

In [8]:
FORMATS = [
    ("yyyy-MM-dd", r"^\d{4}-\d{2}-\d{2}$"),   # standard (target format)
    ("yyyy/MM/dd", r"^\d{4}/\d{2}/\d{2}$"),
    ("dd/MM/yyyy", r"^\d{2}/\d{2}/\d{4}$"),
    ("MM-dd-yyyy", r"^\d{2}-\d{2}-\d{4}$"),
]

def parse_date(col):
    c = F.trim(col)
    # chained when(...).when(...): one value per row
    e = F.when(c.rlike(FORMATS[0][1]), F.to_date(c, FORMATS[0][0]))
    for fmt, rx in FORMATS[1:]:
        e = e.when(c.rlike(rx), F.to_date(c, fmt))
    return e                                   # no match / impossible date -> null

fmt_label = F.when(F.col("order_date").isNull(), "missing")
for fmt, rx in FORMATS:
    fmt_label = fmt_label.when(F.trim("order_date").rlike(rx), fmt)
fmt_label = fmt_label.otherwise("unrecognized")

print("Date formats present:")
bronze.groupBy(fmt_label.alias("format")).count().orderBy(F.desc("count")).show()

print("Dates that cannot be parsed into a real calendar date:")
(bronze.withColumn("parsed", parse_date(F.col("order_date")))
       .where(F.col("order_date").isNotNull() & F.col("parsed").isNull())
       .select("order_id", "order_date").show(truncate=False))

print("Valid date range:")
bronze.select(F.min(parse_date(F.col("order_date"))).alias("min_date"),
              F.max(parse_date(F.col("order_date"))).alias("max_date")).show()

Date formats present:


+------------+-----+
|      format|count|
+------------+-----+
|  yyyy-MM-dd|  996|
|unrecognized|    1|
|  yyyy/MM/dd|    1|
|  MM-dd-yyyy|    1|
|  dd/MM/yyyy|    1|
+------------+-----+

Dates that cannot be parsed into a real calendar date:


+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
+--------+----------+

Valid date range:


+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-01-01|2026-09-30|
+----------+----------+



## 3. Preprocessing → Silver layer

| # | Operation | Method |
|---|-----------|--------|
| 1 | Duplicates | Drop exact duplicate rows |
| 2 | Missing values | Blank → null; text fields filled with `"Unknown"` |
| 3 | Text | Trim, collapse repeated spaces; Title Case for names, category, city, state; upper-case IDs |
| 4 | Numeric validation | Invalid `quantity` (≤0), `unit_price` (≤0), `discount_percent` (<0 or >100) are replaced: quantity → median of valid values, unit price → median valid price of the same product, discount → 0 |
| 5 | Dates | Parse the known formats into `yyyy-MM-dd`; impossible/unrecognized dates become null |
| 6 | Preservation | No rows are dropped except exact duplicates; a `dq_flags` column records every change |

### Step 1: Remove exact duplicates

In [9]:
step1 = bronze.dropDuplicates()
print("Rows before:", raw_count, "| after:", step1.count(), "| removed:", raw_count - step1.count())

Rows before: 1000 | after: 990 | removed: 10


### Step 2: Standardize text and treat blanks as missing

In [10]:
def clean_text(c):
    t = F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")
    return F.when(t == "", None).otherwise(t)

title_cols = ["customer_name", "category", "city", "state"]       # safe to Title Case
plain_cols = ["product", "payment_method", "order_status"]        # keep spelling (e.g. 'USB-C Hub', 'UPI')

step2 = step1
for c in title_cols:
    step2 = step2.withColumn(c, F.initcap(clean_text(c)))
for c in plain_cols:
    step2 = step2.withColumn(c, clean_text(c))
step2 = (step2.withColumn("customer_id", F.upper(clean_text("customer_id")))
              .withColumn("order_id", F.col("order_id").cast("int")))

print("Categories after standardization:")
step2.groupBy("category").count().orderBy("category").show()
print("Cities after standardization:")
step2.groupBy("city").count().orderBy("city").show(30)

Categories after standardization:


+---------------+-----+
|       category|count|
+---------------+-----+
|         Beauty|  137|
|          Books|  127|
|       Clothing|  108|
|    Electronics|  104|
|      Furniture|  129|
|        Grocery|  134|
|Home Appliances|  113|
|         Sports|  138|
+---------------+-----+

Cities after standardization:


+----------+-----+
|      city|count|
+----------+-----+
|      NULL|    8|
| Bengaluru|   90|
|   Chennai|   88|
|Coimbatore|   81|
|     Delhi|   83|
| Hyderabad|   97|
|    Jaipur|   83|
|   Kolkata|   81|
|   Lucknow|   89|
|    Mumbai|  105|
|      Pune|   81|
|Vijayawada|  104|
+----------+-----+



### Step 3: Validate numeric columns

In [11]:
step3 = (step2
    .withColumn("quantity_raw",   F.col("quantity").cast("int"))
    .withColumn("unit_price_raw", F.col("unit_price").cast("double"))
    .withColumn("discount_raw",   F.col("discount_percent").cast("double"))
    .withColumn("q_ok", F.coalesce(F.col("quantity_raw") >= 1, F.lit(False)))          # null / non-numeric -> invalid
    .withColumn("p_ok", F.coalesce(F.col("unit_price_raw") > 0, F.lit(False)))
    .withColumn("d_ok", F.coalesce(F.col("discount_raw").between(0, 100), F.lit(False))))

# Replacement statistics are computed only from VALID values
global_q_median = step3.where("q_ok").approxQuantile("quantity_raw", [0.5], 0.0)[0]
global_p_median = step3.where("p_ok").approxQuantile("unit_price_raw", [0.5], 0.0)[0]
product_median = (step3.where("p_ok").groupBy("product")
                  .agg(F.percentile_approx("unit_price_raw", 0.5).alias("product_median_price")))
print("Median valid quantity:", global_q_median, "| median valid price:", global_p_median)

step3 = (step3.join(F.broadcast(product_median), "product", "left")
    .withColumn("quantity",
        F.when(F.col("q_ok"), F.col("quantity_raw")).otherwise(F.lit(global_q_median)).cast("int"))
    .withColumn("unit_price",
        F.when(F.col("p_ok"), F.col("unit_price_raw"))
         .otherwise(F.coalesce(F.col("product_median_price"), F.lit(global_p_median))).cast("double"))
    .withColumn("discount_percent",
        F.when(F.col("d_ok"), F.col("discount_raw")).otherwise(F.lit(0.0))))

step3.where("NOT q_ok OR NOT p_ok OR NOT d_ok") \
     .select("order_id", "product", "quantity_raw", "quantity", "unit_price_raw", "unit_price",
             "discount_raw", "discount_percent").show(12)

Median valid quantity: 5.0 | median valid price: 1850.0


+--------+--------------------+------------+--------+--------------+----------+------------+----------------+
|order_id|             product|quantity_raw|quantity|unit_price_raw|unit_price|discount_raw|discount_percent|
+--------+--------------------+------------+--------+--------------+----------+------------+----------------+
|  100816|          Smartphone|           3|       3|           0.0|   24760.0|        10.0|            10.0|
|  100857|       Bedside Table|           4|       4|        2460.0|    2460.0|       120.0|             0.0|
|  100629|        Office Chair|           9|       9|           0.0|   11290.0|        15.0|            15.0|
|  100864|Competitive Exam ...|          -2|       5|         970.0|     970.0|        30.0|            30.0|
|  100170|Competitive Exam ...|           6|       6|           0.0|     740.0|         0.0|             0.0|
|  100143|            Football|           0|       5|        1030.0|    1030.0|        15.0|            15.0|
|  100365|

### Step 4: Standardize dates, fill missing text, add data-quality flags

In [12]:
step4 = (step3
    .withColumn("order_date_raw", F.col("order_date"))
    .withColumn("order_date", parse_date(F.col("order_date")))
    .withColumn("order_date", F.date_format("order_date", "yyyy-MM-dd")))   # one consistent format

flags = F.concat_ws(";",
    F.when(F.col("customer_name").isNull(),   "missing_customer_name"),
    F.when(F.col("payment_method").isNull(),  "missing_payment_method"),
    F.when(F.col("city").isNull(),            "missing_city"),
    F.when(~F.col("q_ok"),                    "quantity_replaced"),
    F.when(~F.col("p_ok"),                    "unit_price_replaced"),
    F.when(~F.col("d_ok"),                    "discount_replaced"),
    F.when(F.col("order_date").isNull(),      "invalid_date_set_null"))

silver = (step4
    .withColumn("dq_flags", F.when(flags == "", None).otherwise(flags))
    .fillna({"customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})
    .select("order_id", "customer_id", "customer_name", "product", "category",
            "quantity", "unit_price", "discount_percent", "payment_method",
            "city", "state", "order_date", "order_status", "dq_flags")
    .orderBy("order_id"))

silver.printSchema()
silver.show(8, truncate=False)
print("Rows flagged with at least one fix:", silver.where("dq_flags IS NOT NULL").count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- dq_flags: string (nullable = true)



+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_flags|
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------+
|100001  |C0256      |Keerthi Mishra |Saree        |Clothing |2       |6710.0    |20.0            |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |NULL    |
|100002  |C0042      |Arjun Singh    |T-Shirt      |Clothing |6       |680.0     |5.0             |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |NULL    |
|100003  |C0348      |Keerthi Rao    |Hoodie       |Clothing |6       |3080.0    |12.0            |UPI             |Pune

Rows flagged with at least one fix: 58


## 4. Save to Silver and verify

In [13]:
import glob

# (a) Parquet inside silver/preprocessed_sales/
silver.write.mode("overwrite").parquet(SILVER_PARQUET)

# (b) Single CSV file: silver/sales_silver.csv
tmp = f"{SILVER}/_csv_tmp"
shutil.rmtree(tmp, ignore_errors=True)
silver.coalesce(1).write.mode("overwrite").option("header", True).csv(tmp)
shutil.copyfile(glob.glob(f"{tmp}/part-*.csv")[0], f"{SILVER}/sales_silver.csv")
shutil.rmtree(tmp)

for f in glob.glob(f"{SILVER_PARQUET}/.*.crc"):          # drop Spark's hidden checksum files
    os.remove(f)

for root, dirs, fs in os.walk(BASE):
    print(root + "/", fs)

data_lake/ []
data_lake/bronze/ ['sales.csv']
data_lake/silver/ ['sales_silver.csv']
data_lake/silver/preprocessed_sales/ ['part-00000-614328fc-6c2b-406a-8902-12862784cb01-c000.snappy.parquet', '_SUCCESS']


In [14]:
silver_back = spark.read.parquet(SILVER_PARQUET)        # read back with Spark
silver_count = silver_back.count()

print("Row count BEFORE preprocessing (Bronze):", raw_count)
print("Row count AFTER  preprocessing (Silver):", silver_count)
print("Rows removed:", raw_count - silver_count)

silver_back.show(10, truncate=False)

csv_back = spark.read.option("header", True).csv(f"{SILVER}/sales_silver.csv")
print("CSV read-back count:", csv_back.count())

# Sanity checks on the Silver data (all computed by Spark)
silver_back.select(
    F.sum(F.when(F.col("quantity") < 1, 1).otherwise(0)).alias("quantity_lt_1"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0)).alias("price_le_0"),
    F.sum(F.when(~F.col("discount_percent").between(0, 100), 1).otherwise(0)).alias("discount_out_of_range"),
    F.sum(F.when(F.col("order_date").isNull(), 1).otherwise(0)).alias("null_dates"),
    F.sum(F.when(F.col("order_date").isNotNull() & ~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), 1).otherwise(0)).alias("non_standard_dates"),
    F.countDistinct("category").alias("distinct_categories"),
).show(vertical=True)

Row count BEFORE preprocessing (Bronze): 1000
Row count AFTER  preprocessing (Silver): 990
Rows removed: 10


+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_flags|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710.0    |20.0            |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |NULL    |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680.0     |5.0             |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |NULL    |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3

CSV read-back count: 990


-RECORD 0--------------------
 quantity_lt_1         | 0   
 price_le_0            | 0   
 discount_out_of_range | 0   
 null_dates            | 4   
 non_standard_dates    | 0   
 distinct_categories   | 8   



### Summary of preprocessing operations (counts from Spark)

In [15]:
changed = [F.col(c).isNotNull() & (F.col(c) != F.initcap(clean_text(c))) for c in title_cols]         + [F.col(c).isNotNull() & (F.col(c) != clean_text(c)) for c in plain_cols]         + [F.col("customer_id").isNotNull() & (F.col("customer_id") != F.upper(clean_text("customer_id")))]
any_changed = changed[0]
for cond in changed[1:]:
    any_changed = any_changed | cond
text_fix_rows = step1.where(any_changed).count()          # computed on the de-duplicated raw data

summary = [
    ("Exact duplicate rows removed",                   raw_count - step1.count()),
    ("Missing customer_name filled with 'Unknown'",    silver_back.where(F.col("dq_flags").contains("missing_customer_name")).count()),
    ("Missing payment_method filled with 'Unknown'",   silver_back.where(F.col("dq_flags").contains("missing_payment_method")).count()),
    ("Missing city filled with 'Unknown'",             silver_back.where(F.col("dq_flags").contains("missing_city")).count()),
    ("Invalid quantity replaced (median)",             silver_back.where(F.col("dq_flags").contains("quantity_replaced")).count()),
    ("Invalid unit_price replaced (product median)",   silver_back.where(F.col("dq_flags").contains("unit_price_replaced")).count()),
    ("Invalid discount_percent replaced (0)",          silver_back.where(F.col("dq_flags").contains("discount_replaced")).count()),
    ("Invalid dates set to null",                      silver_back.where(F.col("dq_flags").contains("invalid_date_set_null")).count()),
    ("Non-standard date formats converted to yyyy-MM-dd",
        step4.where(F.col("order_date_raw").isNotNull() & F.col("order_date").isNotNull() & (F.trim("order_date_raw") != F.col("order_date"))).count()),
    ("Rows with whitespace/capitalization fixes in text fields", text_fix_rows),
]
spark.createDataFrame(summary, ["operation", "rows_affected"]).show(truncate=False)

+--------------------------------------------------------+-------------+
|operation                                               |rows_affected|
+--------------------------------------------------------+-------------+
|Exact duplicate rows removed                            |10           |
|Missing customer_name filled with 'Unknown'             |10           |
|Missing payment_method filled with 'Unknown'            |8            |
|Missing city filled with 'Unknown'                      |8            |
|Invalid quantity replaced (median)                      |12           |
|Invalid unit_price replaced (product median)            |8            |
|Invalid discount_percent replaced (0)                   |8            |
|Invalid dates set to null                               |4            |
|Non-standard date formats converted to yyyy-MM-dd       |2            |
|Rows with whitespace/capitalization fixes in text fields|18           |
+--------------------------------------------------

**Preprocessing performed:** exact-duplicate removal; blank/missing text values set to `Unknown`; whitespace trimmed and capitalization standardized; invalid quantity, unit price and discount values replaced with reasonable statistics (and flagged); multiple date formats converted to `yyyy-MM-dd` with impossible dates set to null; all other records preserved. The Bronze file was never modified.